In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

## SILVER LAYER SCRIPT

#### DATA ACCESS USING APP
(https://learn.microsoft.com/en-us/azure/databricks/connect/storage/azure-storage)

In [0]:
# ============================================================
# CURRENT APPROACH (Unity Catalog + Managed Identity)
# ============================================================

display(
    dbutils.fs.ls(
        "abfss://bronze@awstoragedtalake.dfs.core.windows.net/"
    )
)

dbutils.fs.ls("/")

In [0]:
# ============================================================
# DEPRECATED - Spark Configuration Based ADLS Authentication
#
# This approach uses service principal / storage account
# configurations through spark.conf.set().
#
# Not recommended for Unity Catalog-enabled Serverless
# Databricks workspaces.
#
# Replaced by:
# Access Connector + Managed Identity +
# Unity Catalog Storage Credential +
# External Location
#
# Date deprecated: Nov 2026
# ============================================================

# spark.conf.set(
#     "fs.azure.account.auth.type.awstoragedtalake.dfs.core.windows.net",
#     "OAuth"
# )
#
# spark.conf.set(
#     "fs.azure.account.oauth.provider.type.awstoragedtalake.dfs.core.windows.net",
#     "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
# )
#
# spark.conf.set(
#     "fs.azure.account.oauth2.client.id.awstoragedtalake.dfs.core.windows.net",
#     client_id
# )
#
# spark.conf.set(
#     "fs.azure.account.oauth2.client.secret.awstoragedtalake.dfs.core.windows.net",
#     client_secret
# )
#
# spark.conf.set(
#     "fs.azure.account.oauth2.client.endpoint.awstoragedtalake.dfs.core.windows.net",
#     f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
# )



#### DATA LOADING

####Read Data

In [0]:
df_cal = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Calendar")

In [0]:
df_cus = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Customers")

In [0]:
df_procat = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Product_Categories")

In [0]:
df_prosubcat = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Product_Subcategories")

In [0]:
df_prd = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Products")

In [0]:
df_ret = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Returns")

In [0]:
df_sales = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Sales")

In [0]:
df_ter = spark.read.format('csv')\
        .option("header",True)\
            .option("inferSchema",True)\
                .load("abfss://bronze@awstoragedtalake.dfs.core.windows.net/Territories")

##TRANSFORMATIONS

####Calendar

In [0]:
df_cal.display()

In [0]:
df_cal = df_cal.withColumn('Month',month(col('Date')))\
            .withColumn('Year',year(col('Date')))
df_cal.display()

In [0]:
df_cal.write.format('parquet')\
    .option("header",True)\
        .mode("append")\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/Calendar")\
                .save()

In [0]:
display(
    dbutils.fs.ls(
        "abfss://silver@awstoragedtalake.dfs.core.windows.net/"
    )
)

### Customer

In [0]:
df_cus.display()

In [0]:
display(df_cus.withColumn('Fullname',concat(col('Prefix'),col('FirstName'),lit(' ') ,col('LastName'))))

In [0]:
df_cus = df_cus.withColumn('FullName',concat_ws(' ',col('Prefix'),col('FirstName'),col('LastName')))

In [0]:
display(df_cus)

In [0]:
df_cus.write.format('parquet')\
    .option("header",True)\
        .mode("append")\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/Customer")\
                .save()

### Sub Categories

In [0]:
df_prosubcat.write.format('parquet')\
    .option("header",True)\
        .mode("append")\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/ProductSubcategory")\
                .save()

In [0]:
df_prd = df_prd.withColumn('ProductSKU',split(col('ProductSKU'),'-')[0])\
                 .withColumn('ProductName',split(col('ProductName'),' ')[0])


In [0]:
display(df_prd)

In [0]:
df_prd.write.format('parquet')\
    .option("header",True)\
        .mode("append")\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/Products")\
                .save()

### Returns

In [0]:
df_ret.write.format('parquet')\
    .option("header",True)\
        .mode("append")\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/Returns")\
                .save()

### Territories

In [0]:
display(df_ter)

In [0]:
df_ter.write.format('parquet')\
    .option("header",True)\
        .mode("append")\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/Territories")\
                .save()

### Sales

In [0]:
df_sales = df_sales.withColumn('StockDate',to_timestamp('StockDate'))

In [0]:
df_sales = df_sales.withColumn('OrderNumber',regexp_replace(col('OrderNumber'),'S','T'))

In [0]:
df_sales = df_sales.withColumn('multiply',col('OrderLineItem')*col("OrderQuantity"))

In [0]:
display(df_sales)

In [0]:
df_sales.write.format("Parquet")\
    .mode("append")\
        .option("header",True)\
            .option("path","abfss://silver@awstoragedtalake.dfs.core.windows.net/Sales")\
                .save()

### Sales Analysis

In [0]:
df_sales.groupBy('OrderDate').agg(count('OrderNumber').alias('total_order')).display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_procat.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_ter.display()

Databricks visualization. Run in Databricks to view.